<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/02_vector_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install sentence-transformers

In [4]:
import json
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

chunks = load_jsonl(WORKDIR / "document_chunks.jsonl")
questions = load_jsonl(WORKDIR / "questions.jsonl")
print(len(chunks), "chunks,", len(questions), "questions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
182 chunks, 20 questions


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding matrix shape: ", emb.shape)
np.save(WORKDIR / "emb.npy", emb)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape:  (182, 384)


In [6]:
def retrieve(question, k=5):
    q_emb = model.encode([question], normalize_embeddings=True)[0]
    scores = emb @ q_emb
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

def show(q, k=5):
    print(f"\n[{q['type']}] {q['question']}")
    print("Answer:", q["answer"], "| gold paragraphs:", q["gold_titles"])
    for rank, (i, s) in enumerate(retrieve(q["question"], k), 1):
        title = chunks[i]["title"]
        mark = "ok" if title in q["gold_titles"] else "  "
        print(f"  {rank}. {mark} {s:.3f}  {title}")

In [7]:
first_of_type = {}
for q in questions:
    first_of_type.setdefault(q["type"], q)

for q in first_of_type.values():
    show(q, k=5)


[bridge_comparison] Which film has the director who was born first, Slunce, Seno, Jahody or To Sleep With Anger?
Answer: To Sleep With Anger | gold paragraphs: ['Charles Burnett (director)', 'Slunce, seno, jahody', 'To Sleep with Anger', 'Zdeněk Troška']
  1. ok 0.648  Slunce, seno, jahody
  2.    0.577  Slunce, seno, erotika
  3. ok 0.569  To Sleep with Anger
  4.    0.507  Marie Pilátová
  5.    0.486  Slunce, seno a pár facek

[comparison] Who was born earlier, Geoff Bagnall or Caroline Huppert?
Answer: Caroline Huppert | gold paragraphs: ['Caroline Huppert', 'Geoff Bagnall']
  1. ok 0.589  Geoff Bagnall
  2. ok 0.544  Caroline Huppert
  3.    0.379  Tom Dickinson
  4.    0.364  Caroline Robbins
  5.    0.335  Ben Palmer

[compositional] Where was the place of death of Naïma Ben Ali's husband?
Answer: Jeddah | gold paragraphs: ['Naïma Ben Ali', 'Zine El Abidine Ben Ali']
  1. ok 0.631  Naïma Ben Ali
  2.    0.485  Leïla Ben Ali
  3.    0.471  Ibrahim Ben Ali
  4.    0.444  Belhasse